# Fine-tune Mistral-7B with QLoRA for Review Summarization

This notebook fine-tunes Mistral-7B-Instruct using QLoRA (4-bit quantization + LoRA adapters) on 200 product review summary pairs.

**Requirements:** Run on Google Colab with T4 GPU (free tier).

**Steps:**
1. Upload `summary_training_pairs.jsonl` from `data/processed/`
2. Install dependencies
3. Load and format training data
4. Load Mistral-7B in 4-bit
5. Apply LoRA adapters
6. Train for 3 epochs
7. Save adapter weights
8. Test generation
9. Download adapter weights

In [1]:
# Step 1: Upload training data
from google.colab import files
uploaded = files.upload()  # Upload summary_training_pairs.jsonl

Saving summary_training_pairs.jsonl to summary_training_pairs.jsonl


In [2]:
# Step 2: Install dependencies
!pip install -q transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.8 MB/s eta 0:00:00


In [4]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


In [5]:
# Step 3: Load and format training data
pairs = []
with open('summary_training_pairs.jsonl') as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} training pairs")

# Format as chat messages for Mistral-Instruct
def format_prompt(pair):
    return (
        f"<s>[INST] You are an expert product analyst. "
        f"Given the following customer reviews, write a structured executive summary.\n\n"
        f"{pair['input'][:3000]}\n[/INST]\n"
        f"{pair['summary']}</s>"
    )

formatted = [{'text': format_prompt(p)} for p in pairs]
dataset = Dataset.from_list(formatted)

# Train/val split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
val_ds = split['test']
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"\nSample:\n{formatted[0]['text'][:500]}...")

Loaded 200 training pairs
Train: 180, Val: 20

Sample:
<s>[INST] You are an expert product analyst. Given the following customer reviews, write a structured executive summary.

Product: Senso Bluetooth Headphones, Best Wireless Sports Earbuds w/Mic IPX7 Waterproof HD Stereo Sweatproof Earphones for Gym Running Workout Noise Cancelling Earphones Earbuds Noise Cancelling Headsets

Total reviews shown: 50

Negative (1-2 stars): 32 | Positive (4-5 stars): 12

---

[5 stars] Battery issues resolved through a reset
Great sound quality and I enjoyed them u...


In [6]:
# Step 4: Load Mistral-7B in 4-bit quantization
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model loaded: mistralai/Mistral-7B-Instruct-v0.2
Parameters: 7.2B


In [7]:
# Step 5: Apply LoRA adapters
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% of total parameters are trainable

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [14]:
# Step 6: Train for 3 epochs                                                                                                          training_args = TrainingArguments(
training_args = TrainingArguments(
      output_dir="./mistral-qlora-reviews",
      num_train_epochs=3,
      per_device_train_batch_size=2,
      per_device_eval_batch_size=2,
      gradient_accumulation_steps=4,
      learning_rate=2e-4,
      weight_decay=0.01,
      warmup_steps=20,
      logging_steps=10,
      eval_strategy="epoch",
      save_strategy="epoch",
      load_best_model_at_end=True,
      bf16=True,
      report_to="none",
      optim="paged_adamw_8bit",
      max_grad_norm=0.3,
      lr_scheduler_type="cosine",
)
trainer = SFTTrainer(
      model=model,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      processing_class=tokenizer,
      args=training_args,
  )
print("Starting training...")
trainer.train()
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.972436,1.850176
2,1.839088,1.838560
3,1.738666,1.836589


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete!


In [15]:
# Step 7: Save adapter weights
ADAPTER_DIR = "./mistral-qlora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

# Check adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6
print(f"Adapter size: {total_size:.1f} MB")

Adapter saved to ./mistral-qlora-adapter
Adapter size: 58.1 MB


In [16]:
# Step 8: Test generation — compare base vs fine-tuned
test_input = pairs[-1]['input'][:2000]  # Use last pair as test

prompt = (
    f"<s>[INST] You are an expert product analyst. "
    f"Given the following customer reviews, write a structured executive summary.\n\n"
    f"{test_input}\n[/INST]\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract just the response (after [/INST])
response = generated.split('[/INST]')[-1].strip()
print("=== Fine-tuned Mistral Output ===")
print(response)

=== Fine-tuned Mistral Output ===
## Product Issues Summary

**Overall Sentiment:** Positive ([92]% negative reviews)

**Top Complaints:**
1. **Driver Issues** — 2 mentions — Some customers experienced issues with outdated drivers, which rendered the product useless without activation.
2. **Short Lifespan** — 1 mention — One customer reported that the product stopped working after a few days.
3. **No Instructions** — 1 mention — A customer mentioned that there were no instructions included with the product.

**Key Insights:**
- The majority of customers (92%) were satisfied with the product, citing easy setup and compatibility with various devices.
- The product's range and performance were also praised by customers, with some reporting a range of up to 30 feet.
- The product's compatibility with Windows 11, 10, 8.x, 7, Classic Bluetooth, gamepad, and stereo headset was a significant selling point for customers.

**Recommendations:**
- The manufacturer should ensure that drivers are up

In [17]:
# Step 9: Download adapter weights
!zip -r mistral-qlora-adapter.zip mistral-qlora-adapter/
files.download('mistral-qlora-adapter.zip')
print("\nDownload the zip and extract to models/mistral-qlora/ in your project")

  adding: mistral-qlora-adapter/ (stored 0%)
  adding: mistral-qlora-adapter/tokenizer_config.json (deflated 48%)
  adding: mistral-qlora-adapter/chat_template.jinja (deflated 64%)
  adding: mistral-qlora-adapter/tokenizer.json (deflated 85%)
  adding: mistral-qlora-adapter/README.md (deflated 65%)
  adding: mistral-qlora-adapter/adapter_config.json (deflated 57%)
  adding: mistral-qlora-adapter/adapter_model.safetensors (deflated 54%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download the zip and extract to models/mistral-qlora/ in your project


In [18]:
# Step 10: Also save training loss for documentation
history = trainer.state.log_history
print("\n=== Training History ===")
for entry in history:
    if 'loss' in entry:
        print(f"Step {entry.get('step', '?')}: loss={entry['loss']:.4f}")
    if 'eval_loss' in entry:
        print(f"  eval_loss={entry['eval_loss']:.4f}")


=== Training History ===
Step 10: loss=2.3630
Step 20: loss=1.9724
  eval_loss=1.8502
Step 30: loss=1.8348
Step 40: loss=1.8391
  eval_loss=1.8386
Step 50: loss=1.7878
Step 60: loss=1.7387
  eval_loss=1.8366
